# 🎧 Visualisation de données — Spotify Tracks Dataset
**UE Visualisation des données** · Groupe : _à compléter_ · Membres : _à compléter_

Ce notebook présente le projet, le public visé, les objectifs et les tâches d'analyse. Chaque tâche est accompagnée d'une courte vérification sur les données réelles pour montrer qu'elle est réalisable.

## 1. Présentation du projet

Spotify décrit chaque morceau de son catalogue par une série de **caractéristiques audio** calculées automatiquement : *danceability*, *energy*, *valence* (positivité), *acousticness*, *tempo*, *loudness*, etc. Ces mesures permettent de comparer des morceaux de façon objective, au-delà de l'étiquette de genre.

Notre jeu de données (*Spotify Tracks Dataset*, Kaggle) contient **114 000 morceaux** répartis sur **114 genres** (1 000 morceaux par genre), décrits par **21 attributs** :

- **identification** : `track_id`, `track_name`, `artists`, `album_name`, `track_genre` ;
- **audience** : `popularity` (score de 0 à 100) ;
- **caractéristiques audio continues** : `danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `duration_ms` ;
- **attributs catégoriels** : `explicit`, `key`, `mode` (majeur/mineur), `time_signature`.

Il s'agit donc d'un jeu **multivarié**, mêlant attributs quantitatifs, ordinaux et nominaux, adapté à des visualisations interactives de type *overview + detail* : partir d'une vue d'ensemble des genres, puis descendre jusqu'à l'artiste ou au morceau.

L'objectif du projet est de construire une interface qui aide à **comprendre ce qui distingue musicalement les genres** et **quelles caractéristiques sont associées à la popularité**. La préparation des données comprendra notamment : suppression des valeurs manquantes et des doublons (un même morceau peut apparaître dans plusieurs genres), conversion de la durée en minutes, calcul de profils moyens par genre et par artiste, et regroupement de genres proches.

In [ ]:
import pandas as pd
df = pd.read_csv("dataset.csv", index_col=0)
print(df.shape)
print("Genres :", df["track_genre"].nunique(), "| Artistes :", df["artists"].nunique())
print("Valeurs manquantes :", int(df.isna().sum().sum()),
      "| track_id présents dans plusieurs lignes :", int(df["track_id"].duplicated().sum()))
df.head()

(114000, 20)
Genres : 114 | Artistes : 31437
Valeurs manquantes : 3 | track_id présents dans plusieurs lignes : 24259


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 2. Public ciblé

**Curateurs de playlists et programmateurs musicaux** : éditeurs de playlists sur les plateformes de streaming, programmateurs radio, DJs et organisateurs d'événements.

**Pourquoi ce public ?**
- Leur métier consiste à **assembler des morceaux cohérents** (une ambiance, un rythme, une énergie) et à **anticiper ce qui plaira** au public.
- Ils raisonnent déjà avec des notions proches des attributs du jeu de données : tempo (BPM), énergie, ambiance joyeuse ou mélancolique, contenu explicite.
- Ils ont besoin de passer d'une vue large (« quels genres conviennent à une soirée dansante ? ») à une sélection précise (« quels morceaux à 120–130 BPM, très dansants et peu connus ? »), ce qui correspond au principe *overview + detail*.

**Profil d'usage** : utilisateurs non spécialistes de la statistique, à l'aise avec la musique, qui veulent explorer rapidement, filtrer et comparer.

## 3. Objectifs

| # | Objectif | Question à laquelle l'outil répond |
|---|----------|------------------------------------|
| O1 | **Caractériser les genres** | Quel est le profil sonore de chaque genre et quels genres se ressemblent ? |
| O2 | **Comprendre la popularité** | Quelles caractéristiques audio sont associées aux morceaux les plus écoutés ? |
| O3 | **Construire une sélection** | Comment trouver rapidement des morceaux répondant à une ambiance donnée ? |
| O4 | **Repérer l'atypique** | Quels morceaux ou artistes sortent du profil habituel de leur genre ? |

## 4. Tâches d'analyse

Les tâches sont formulées selon la typologie de Munzner (*action + cible*). Chacune est vérifiée ci-dessous sur les données réelles.

| Tâche | Objectif | Type | Attributs utilisés |
|-------|----------|------|--------------------|
| **T1** – Comparer le profil audio moyen de plusieurs genres | O1 | Comparer | `track_genre` + 7 caractéristiques audio |
| **T2** – Identifier les genres les plus proches d'un genre donné | O1 | Identifier / similarité | profils moyens par genre |
| **T3** – Observer la distribution de la popularité par genre | O2 | Résumer / distribution | `popularity`, `track_genre` |
| **T4** – Mettre en relation les caractéristiques audio et la popularité | O2 | Corréler | `popularity` × caractéristiques |
| **T5** – Comparer la part de contenu explicite selon les genres | O2 | Comparer | `explicit`, `track_genre` |
| **T6** – Filtrer les morceaux selon des plages de tempo, énergie, valence | O3 | Filtrer / explorer | `tempo`, `energy`, `valence`, `danceability` |
| **T7** – Consulter le détail d'un artiste ou d'un morceau sélectionné | O3 | Consulter (*detail*) | `artists`, `track_name`, `album_name`, toutes les mesures |
| **T8** – Localiser les morceaux atypiques au sein de leur genre | O4 | Localiser les valeurs aberrantes | écart au profil moyen du genre |

In [ ]:
features = ["danceability", "energy", "valence", "acousticness",
            "speechiness", "instrumentalness", "liveness"]
clean = df.dropna().drop_duplicates(subset=["track_id", "track_genre"]).copy()
clean["duration_min"] = clean["duration_ms"] / 60000
profils = clean.groupby("track_genre")[features + ["tempo", "popularity"]].mean()

### Vérification T1 — profils moyens de quelques genres

In [ ]:
profils.loc[["classical", "edm", "hip-hop", "jazz", "metal", "sad"], features].round(2)

,danceability,energy,valence,acousticness,speechiness,instrumentalness,liveness
track_genre,,,,,,,
classical,0.38,0.20,0.38,0.92,0.05,0.61,0.17
edm,0.65,0.76,0.46,0.11,0.07,0.04,0.19
hip-hop,0.74,0.68,0.55,0.20,0.13,0.01,0.19
jazz,0.51,0.35,0.49,0.72,0.05,0.08,0.17
metal,0.46,0.84,0.42,0.04,0.07,0.06,0.21
sad,0.69,0.46,0.42,0.47,0.13,0.11,0.17


### Vérification T2 — genres les plus proches de « house » (distance euclidienne sur les profils)

In [ ]:
z = (profils[features] - profils[features].mean()) / profils[features].std()
dist = ((z - z.loc["house"]) ** 2).sum(axis=1) ** 0.5
dist.drop("house").nsmallest(5).round(2)

track_genre
edm            0.28
dance          0.66
electro        0.79
synth-pop      0.91
alternative    1.02
dtype: float64

### Vérification T3 — genres les plus et les moins populaires (médiane)

In [ ]:
pop = clean.groupby("track_genre")["popularity"].median().sort_values()
pd.concat([pop.tail(5), pop.head(5)]).to_frame("popularité médiane")

,popularité médiane
track_genre,
chill,57.0
hip-hop,58.0
k-pop,60.0
pop-film,60.0
pop,66.0
country,0.0
iranian,0.0
latin,0.0
soul,0.0


> **À noter :** certains genres ont une popularité médiane de 0. Ce point devra être traité dans la préparation des données (morceaux jamais écoutés ou valeurs manquantes codées 0).

### Vérification T4 — corrélation des caractéristiques avec la popularité

In [ ]:
clean[features + ["tempo", "loudness", "duration_min"]].corrwith(clean["popularity"], method="spearman").sort_values().round(3)

instrumentalness   -0.078
speechiness        -0.068
valence            -0.042
energy             -0.027
liveness           -0.008
acousticness        0.011
tempo               0.016
danceability        0.026
duration_min        0.027
loudness            0.033
dtype: float64

> **À noter :** sur l'ensemble du catalogue, les corrélations sont faibles. La popularité dépend donc surtout du genre : la visualisation devra permettre d'observer ces relations **genre par genre**, ce qui est l'intérêt d'une vue interactive.

### Vérification T5 — part de morceaux explicites par genre

In [ ]:
(clean.groupby("track_genre")["explicit"].mean() * 100).sort_values(ascending=False).head(8).round(1).to_frame("% explicite")

,% explicite
track_genre,
comedy,65.6
emo,46.5
sad,45.0
j-dance,38.9
hardcore,32.5
hip-hop,31.7
funk,30.4
dancehall,30.2


### Vérification T6 — exemple de filtre : « soirée dansante » (120–130 BPM, énergie > 0,8, danceability > 0,7)

In [ ]:
sel = clean[clean["tempo"].between(120, 130) & (clean["energy"] > 0.8) & (clean["danceability"] > 0.7)]
print(sel["track_id"].nunique(), "morceaux distincts correspondent")
sel.drop_duplicates("track_id").sort_values("popularity", ascending=False)[["track_name", "artists", "track_genre", "tempo", "popularity"]].head(5)

1850 morceaux distincts correspondent


,track_name,artists,track_genre,tempo,popularity
20014,One Kiss (with Dua Lipa),Calvin Harris;Dua Lipa,dance,123.994,89
81354,Bad Habits,Ed Sheeran,pop,126.011,88
20030,Bad Decisions (with BTS & Snoop Dogg),benny blanco;BTS;Snoop Dogg,dance,120.048,87
20028,There's Nothing Holdin' Me Back,Shawn Mendes,dance,121.998,86
30458,Hot In It,Tiësto;Charli XCX,edm,123.045,84


### Vérification T7 — fiche détail d'un artiste

In [ ]:
artiste = clean["artists"].value_counts().index[0]
print("Artiste :", artiste)
clean[clean["artists"] == artiste][["track_name", "album_name", "track_genre", "popularity"] + features[:3]].head(5)

Artiste : The Beatles


,track_name,album_name,track_genre,popularity,danceability,energy,valence
11016,Here Comes The Sun - Remastered 2009,Abbey Road (Remastered),british,82,0.557,0.540,0.394
11017,Yesterday - Remastered 2009,Help! (Remastered),british,76,0.332,0.179,0.315
11018,Let It Be - Remastered 2009,Let It Be (Remastered),british,77,0.443,0.403,0.410
11052,Hey Jude,Love,british,66,0.462,0.632,0.585
11055,Come Together - Remastered 2009,Abbey Road (Remastered),british,77,0.533,0.376,0.187


### Vérification T8 — morceaux les plus atypiques de leur genre (écart au profil moyen)

In [ ]:
moy = clean.groupby("track_genre")[features].transform("mean")
std = clean.groupby("track_genre")[features].transform("std")
clean["ecart"] = (((clean[features] - moy) / std) ** 2).sum(axis=1) ** 0.5
clean.nlargest(5, "ecart")[["track_name", "artists", "track_genre", "ecart"]].round(2)

,track_name,artists,track_genre,ecart
77569,Alvará - Ao vivo,Jorge Aragão,pagode,30.96
97321,Capricha Gaiteiro,Garotos de Ouro,sertanejo,29.69
78808,I Like To Move It From Madagascar,The Party Cats,party,26.79
35209,Pisadinha dos 600,Turma da Pisadinha;Anderson & Vei da Pisadinha,forro,26.06
4073,Varraaru Vaarraaru Yaaru Varraaru,Mohan;Murali;Chandiran,ambient,24.96


## 5. Suite du projet
- Toutes les tâches ci-dessus sont réalisables avec les attributs présents dans le jeu de données.
- **Étape suivante** : documenter le pipeline de préparation complet (nettoyage, transformations, agrégations) puis, avec l'enseignant, choisir une technique de visualisation interactive différente par membre du groupe.